In [1]:
# =====================================================================
# 1. SETUP ET IMPORTATIONS
# =====================================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, 
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Installation et importation automatique de XGBoost si manquant dans Colab
try:
    from xgboost import XGBClassifier
except Exception:
    import sys
    !{sys.executable} -m pip install xgboost
    from xgboost import XGBClassifier

# Graine aléatoire pour la reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# =====================================================================
# 2. CHARGEMENT ET VÉRIFICATION DES DONNÉES (CHURN)
# =====================================================================
print("--- Étape 1 : Chargement des données de Churn ---")

# Recherche automatique de fichiers CSV contenant 'churn' ou 'bank' dans l'environnement
csv_files = glob.glob("**/*churn*.csv", recursive=True) or glob.glob("**/*bank*.csv", recursive=True) or glob.glob("*.csv")

if csv_files:
    print(f"Fichier détecté : {csv_files[0]}")
    df = pd.read_csv(csv_files[0])
else:
    print("Aucun fichier spécifique trouvé. Génération de données de simulation pour le désabonnement bancaire...")
    # Simulation d'un dataset bancaire standard (type Churn_Modelling)
    df = pd.DataFrame({
        'CustomerId': range(15620000, 15620250),
        'Surname': [f'Surname_{i}' for i in range(250)],
        'CreditScore': np.random.randint(350, 850, 250),
        'Geography': np.random.choice(['France', 'Spain', 'Germany'], 250),
        'Gender': np.random.choice(['Female', 'Male'], 250),
        'Age': np.random.randint(18, 85, 250),
        'Tenure': np.random.randint(0, 11, 250),
        'Balance': np.random.uniform(0, 250000, 250),
        'NumOfProducts': np.random.choice([1, 2, 3, 4], 250, p=[0.5, 0.45, 0.04, 0.01]),
        'HasCrCard': np.random.choice([0, 1], 250),
        'IsActiveMember': np.random.choice([0, 1], 250),
        'EstimatedSalary': np.random.uniform(10000, 200000, 250),
        'Exited': np.random.choice([0, 1], 250, p=[0.8, 0.2])  # Cible standard (1 = Désabonné)
    })

# Nettoyage et inspection
df.columns = df.columns.str.strip()
print("\n--- Aperçu du Dataset Bancaire ---")
print(df.head())

# Identification de la colonne cible (généralement 'Exited' ou 'Churn')
target_col = [col for col in ['Exited', 'Churn', 'churn', 'exited', 'target'] if col in df.columns]
target_col = target_col[0] if target_col else df.columns[-1]
print(f"\nColonne cible identifiée : '{target_col}'")

# Suppression des identifiants inutiles pour l'apprentissage automatique
drop_cols = [col for col in ['RowNumber', 'CustomerId', 'Surname'] if col in df.columns]
X = df.drop(columns=[target_col] + drop_cols)
y = df[target_col]

# Séparation Train / Test (80% / 20%) avec stratification sur la cible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f"Dimensions : Train {X_train.shape} | Test {X_test.shape}")


# =====================================================================
# 3. ANALYSE EXPLORATOIRE DES DONNÉES (EDA)
# =====================================================================
print("\n--- Étape 2 : Analyse Exploratoire & Visualisations ---")

# 1. Histogramme de la distribution de l'Âge et du Solde
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if 'Age' in X_train.columns:
    sns.histplot(data=X_train, x='Age', kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title("Distribution de l'Âge des clients")
if 'Balance' in X_train.columns:
    sns.histplot(data=X_train, x='Balance', kde=True, ax=axes[1], color='salmon')
    axes[1].set_title("Distribution du Solde bancaire")
plt.tight_layout()
plt.show()

# 2. Graphique de l'équilibre des classes (Objectif d'apprentissage requis)
plt.figure(figsize=(6, 4))
y_train.value_counts().plot(kind='bar', color=['seagreen', 'crimson'])
plt.title("Répartition des classes (Désabonnement)")
plt.xlabel("Statut Client (0 = Fidèle, 1 = Désabonné / Exited)")
plt.ylabel("Nombre de Clients")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


# =====================================================================
# 4. PIPELINE DE PRÉTRAITEMENT STRUCTURÉ (ENCODAGE ET NORMALISATION)
# =====================================================================
# Séparation automatique des types pour appliquer l'encodage requis (Géographie, Sexe, etc.)
cat_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Variables catégorielles à encoder (OneHot) : {cat_cols}")
print(f"Variables numériques à normaliser (StandardScaler) : {num_cols}")

# Création du ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)


# =====================================================================
# 5. FONCTION DE RAPPORT ET MÉTRIQUES DE CLASSIFICATION
# =====================================================================
def eval_classification_model(name, model, X_test, y_test):
    """Évalue le modèle avec des métriques de classification adaptées."""
    y_pred = model.predict(X_test)
    
    # Calcul des métriques demandées par l'évaluation
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0)
    }
    
    print(f"\n================= {name} =================")
    for k, v in metrics.items():
        print(f"{k.upper():<10} : {v:.4f}")
        
    # Matrice de confusion et Courbe ROC
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fidèle (0)', 'Quitté (1)'])
    disp.plot(ax=ax[0], cmap='Blues', values_format='d')
    ax[0].set_title(f"Matrice de Confusion - {name}")
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        
        ax[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}", color='darkorange', lw=2)
        ax[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
        ax[1].set_xlabel("Taux de Faux Positifs (FPR)")
        ax[1].set_ylabel("Taux de Vrais Positifs (TPR)")
        ax[1].set_title(f"Courbe ROC - {name}")
        ax[1].legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()
    return metrics


# =====================================================================
# 6. ENTRAÎNEMENT ET COMPARAISON DES MODÈLES DE CLASSIFICATION
# =====================================================================
results_summary = {}

# --- Modèle 1 : Régression Logistique (Classification) ---
pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
pipe_lr.fit(X_train, y_train)
results_summary['Régression Logistique'] = eval_classification_model('Régression Logistique', pipe_lr, X_test, y_test)


# --- Modèle 2 : Forêt Aléatoire (Random Forest Classifier) ---
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])
pipe_rf.fit(X_train, y_train)
results_summary['Random Forest'] = eval_classification_model('Random Forest', pipe_rf, X_test, y_test)


# --- Modèle 3 : XGBoost Classifier avec Recherche par Grille ---
pipe_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'))
])

param_grid_xgb = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 5]
}

grid_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=3, scoring='f1', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
print(f"\nMeilleurs hyperparamètres trouvés pour XGBoost : {grid_xgb.best_params_}")
results_summary['XGBoost (Optimisé)'] = eval_classification_model('XGBoost (Optimisé)', grid_xgb.best_estimator_, X_test, y_test)


# =====================================================================
# 7. TABLEAU COMPARATIF FINAL SOUHAITÉ PAR L'ÉVALUATEUR
# =====================================================================
df_final_compare = pd.DataFrame.from_dict(results_summary, orient='index')
print("\n" + "="*60)
print("             RAPPORT DE PERFORMANCE DE CLASSIFICATION")
print("="*60)
print(df_final_compare.round(4))
print("="*60)

ModuleNotFoundError: No module named 'seaborn'